# **Environment Setup**

In [ ]:
# Environment Setup
from dotenv import load_dotenv
import os
import cassio

# Load environment variables
load_dotenv()
ASTRA_DB_APPLICATION_TOKEN = os.getenv("ASTRA_DB_APPLICATION_TOKEN")
ASTRA_DB_ID = os.getenv("ASTRA_DB_ID")
GROQ_API_KEY = os.getenv("GROQ_API_KEY")  

# Connect to AstraDB
print("Connecting to AstraDB...")
cassio.init(token=ASTRA_DB_APPLICATION_TOKEN, database_id=ASTRA_DB_ID)
print("Connected to AstraDB successfully.")


# **Loading Data from URLs**

In [ ]:
# Data Loading
from langchain_community.document_loaders import WebBaseLoader  # To load web data


urls = [
    'https://lilianweng.github.io/posts/2023-06-23-agent/',
    'https://lilianweng.github.io/posts/2023-03-15-prompt-engineering/',
    'https://lilianweng.github.io/posts/2023-10-25-adv-attack-llm/'
]

# Load the URLs and extract documents
print("Loading documents from URLs...")
docs = [WebBaseLoader(url).load() for url in urls]
doc_list = [item for sublist in docs for item in sublist]  # Flatten the list of lists
print(f"Loaded {len(doc_list)} documents.")


# **Splitting the documents into chunks and creating embeddings**

In [ ]:
# Document Splitting
from langchain.text_splitter import RecursiveCharacterTextSplitter

# Split the documents into smaller chunks
print("Splitting documents into chunks...")
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=500, chunk_overlap=0
)
docs_split = text_splitter.split_documents(doc_list)
print(f"Split documents into {len(docs_split)} chunks.")


In [ ]:
# Embedding Creation
from langchain_huggingface import HuggingFaceEmbeddings

# Create the embeddings object
print("Creating embeddings object...")
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
print("Embeddings object created successfully.")


# **Initialize Vector Store and adding documents**



In [ ]:
# Vector Store Initialization
from langchain.vectorstores.cassandra import Cassandra

# Initialize the vector store backed by AstraDB
print("Initializing AstraDB vector store...")
astra_vector_store = Cassandra(
    embedding=embeddings,
    table_name="qa_mini_demo",  
    session=None,  # `cassio` manages the session internally
    keyspace=None  # `cassio` automatically detects the keyspace
)
print("AstraDB vector store initialized successfully.")


In [ ]:
# Add Documents to Vector Store
print("Adding documents to AstraDB vector store...")
astra_vector_store.add_documents(docs_split)  # Synchronous addition of documents
print(f"Inserted {len(docs_split)} documents into AstraDB vector store.")


In [ ]:
docs_split

# **Query the Vector Store**

In [ ]:
# Query the Vector Store
print("Querying the vector store...")
retriever = astra_vector_store.as_retriever()
query = "What is an agent?"
retrieved_docs = retriever.get_relevant_documents(query)

# Display retrieved documents
if retrieved_docs:
    print(f"Retrieved {len(retrieved_docs)} documents:")
    for i, doc in enumerate(retrieved_docs):
        print(f"Document {i+1}:")
        print(doc.page_content)
else:
    print("No documents retrieved for the query.")


## **Query Router Integration**
 - Router Creation


In [ ]:
# Query Router Integration
from typing import Literal
from langchain_core.pydantic_v1 import BaseModel, Field
from langchain_core.prompts import ChatPromptTemplate
from langchain_groq import ChatGroq

# Data Model for Routing
class RouteQuery(BaseModel):
    """Routes user queries to the most relevant datasource."""
    datasource: Literal["vectorstore", "wikisearch"] = Field(
        ..., description="Route query to either vectorstore or Wikipedia search."
    )

# Initialize the LLM for routing
print("Initializing LLM for routing...")
llm = ChatGroq(groq_proxy=GROQ_API_KEY, model_name="Llama-3.1-70b-Versatile")
structured_llm_router = llm.with_structured_output(RouteQuery)

# Prompt for Routing
system = """You are an expert router for user queries.
Use 'vectorstore' for topics related to agents, prompt engineering, and adversarial attacks.
For all other topics, route the query to 'wikisearch'."""

route_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system),
        ("human", "{question}")
    ]
)

question_router = route_prompt | structured_llm_router
print("LLM-based router initialized successfully.")


# **Router**


In [ ]:
# Router Setup
from typing import List, Dict, Any
from typing_extensions import TypedDict
from langchain.schema import Document
from langchain_community.utilities import WikipediaAPIWrapper
from langchain.vectorstores.cassandra import Cassandra

# Define the Graph State
class GraphState(TypedDict):
    """
    Represents the state of our graph.

    Attributes:
        question (str): The user's question.
        generation (str): Generated response from the LLM.
        documents (List[Document]): List of retrieved documents.
    """
    question: str
    generation: str
    documents: List[Document]

# Initialize Wikipedia Wrapper
api_wrapper = WikipediaAPIWrapper(top_k_results=1, doc_content_chars_max=300)
print("Wikipedia API Wrapper initialized successfully.")


# **Document Retrieval, Wiki Search and Router Fucntion**

In [ ]:
# Document Retrieval Function
def retrieve(state: Dict[str, Any], retriever):
    """
    Retrieve documents from the vector store based on the question.

    Args:
        state (dict): The current graph state.
        retriever: The vector store retriever.

    Returns:
        dict: Updated state with retrieved documents.
    """
    question = state['question']
    print(f"Retrieving documents for question: {question}")
    
    # Retrieve documents
    documents = retriever.invoke(question)
    return {"documents": documents, "question": question}


In [ ]:
from langchain_community.utilities import ArxivAPIWrapper, WikipediaAPIWrapper
from langchain_community.tools import ArxivQueryRun, WikipediaQueryRun
from langchain.schema import Document

# Initialize Arxiv and Wikipedia Wrappers
arxiv_wrapper = ArxivAPIWrapper(top_k_results=1, doc_content_chars_max=200)
arxiv = ArxivQueryRun(api_wrapper=arxiv_wrapper)

api_wrapper = WikipediaAPIWrapper(top_k_results=1, doc_content_chars_max=200)
wiki = WikipediaQueryRun(api_wrapper=api_wrapper)

print("Arxiv and Wikipedia tools initialized successfully.")


In [ ]:
def wiki_search(state):
    """
    Wikipedia search based on the re-phrased question.

    Args:
        state (dict): The current graph state.

    Returns:
        dict: Updated state with Wikipedia results.
    """
    print("---Performing Wikipedia Search---")
    question = state["question"]
    print(f"Search Query: {question}")

    # Perform the Wikipedia search
    try:
        docs = wiki.run(question)  # Use the WikipediaQueryRun tool
        wiki_results = Document(page_content=docs)
        
        # Update the state with the retrieved results
        return {"documents": [wiki_results], "question": question}
    except Exception as e:
        print(f"Error during Wikipedia Search: {e}")
        return {"documents": [], "question": question}


In [ ]:
def route_question(state):
    """
    Route the question to either wiki_search or vectorstore.

    Args:
        state (dict): The current graph state.

    Returns:
        str: The next node to execute.
    """
    print("---Routing Question---")
    question = state["question"]

    try:
        # Invoke the LLM router
        source = structured_llm_router.invoke(question)
        if source.datasource == "wikisearch":
            print("---Routing to Wikipedia Search---")
            return "wiki_search"
        elif source.datasource == "vectorstore":
            print("---Routing to Vector Store---")
            return "vectorstore"
    except Exception as e:
        print(f"Error in routing logic: {e}")
        raise e


# **Langraph Flow**

In [ ]:
from langgraph.graph import StateGraph, START, END

# Initialize StateGraph
workflow = StateGraph(GraphState)

# Add nodes
workflow.add_node("wiki_search", wiki_search)
workflow.add_node("vectorstore", retrieve)

# Add conditional edges using the router
workflow.add_conditional_edges(
    START,
    route_question,  # Use the updated routing function
    {
        "wiki_search": "wiki_search",
        "vectorstore": "vectorstore",
    },
)

# Define Endpoints
workflow.add_edge("wiki_search", END)
workflow.add_edge("vectorstore", END)

# Compile the workflow
app = workflow.compile()
print("Workflow compiled successfully.")


# **Workflow Display**

In [ ]:
from IPython.display import Image, display

# Ensure the graph visualization is generated and displayed
try:
    # Generate the Mermaid diagram as a PNG
    graph_image = app.get_graph().draw_mermaid_png()
    
    # Display the image in the notebook
    display(Image(graph_image))
except Exception as e:
    print(f"An error occurred while generating the graph: {e}")


In [ ]:
from pprint import pprint

# Test input
inputs = {
    "question": "Tell me about Avengers"
}

# Run the workflow and stream the outputs for debugging
try:
    for output in app.stream(inputs):
        for key, value in output.items():
            # Print the current node
            pprint(f"Node '{key}':")
            
            # Optional: Print the state at each node
            pprint(value, indent=2, width=80, depth=None)
        
        # Separate outputs for readability
        pprint("\n---\n")
    
    # Final output: Retrieve the documents from the last node
    final_state = value  # The value from the last node processed
    pprint(final_state["documents"])  # Print the retrieved documents
except Exception as e:
    print(f"Workflow execution failed: {e}")


# **Streamlit APP**

In [ ]:
# Import Streamlit
import streamlit as st

# Streamlit app for querying
def streamlit_frontend():
    st.title("Question Answering App")
    st.write("Ask a question, and we'll tell you if the answer is from the database or Wikipedia.")

    # Input from user
    user_question = st.text_input("Enter your question:", "")

    if st.button("Get Answer"):
        if user_question.strip() == "":
            st.warning("Please enter a valid question.")
            return
        
        # Route the question using the router
        st.write("Routing the question...")
        try:
            source = question_router.invoke({"question": user_question})
            datasource = source.datasource
        except Exception as e:
            st.error(f"Routing failed: {e}")
            return

        # Query the appropriate source
        if datasource == "vectorstore":
            st.write("Fetching from database...")
            retriever = astra_vector_store.as_retriever()
            try:
                docs = retriever.get_relevant_documents(user_question)
                if docs:
                    st.success("Answer from Database:")
                    st.write(docs[0].page_content)  # Display the first retrieved document
                else:
                    st.warning("No relevant documents found in the database.")
            except Exception as e:
                st.error(f"Database query failed: {e}")
        elif datasource == "wikisearch":
            st.write("Fetching from Wikipedia...")
            try:
                wiki_result = wiki.run(user_question)
                st.success("Answer from Wikipedia:")
                st.write(wiki_result)
            except Exception as e:
                st.error(f"Wikipedia query failed: {e}")
        else:
            st.error("Unexpected datasource routing result.")

# Run the Streamlit app
if __name__ == "__main__":
    streamlit_frontend()
